In [30]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from openpmd_viewer import OpenPMDTimeSeries
from scipy.ndimage import gaussian_filter
import scipy.constants as sc

In [31]:
MU0  = sc.mu_0

save_path = './diags'
series_f   = OpenPMDTimeSeries(save_path + '/diag')
iterations = series_f.iterations

# Beta map over time

In [ ]:
# ── Precompute ───────────────────────────────────────────────────
beta_frames = []
xs = zs = None

for k, it in enumerate(iterations[1:]):
    print(f"  Loading {k+1}/{len(iterations)}", end='\r')

    Bx, info = series_f.get_field('B', coord='x', iteration=it, slice_across='y')
    By, _    = series_f.get_field('B', coord='y', iteration=it, slice_across='y')
    Bz, _    = series_f.get_field('B', coord='z', iteration=it, slice_across='y')
    rho, _   = series_f.get_field('rho_stream_i', iteration=it, slice_across='y')
    rho_bg, _ = series_f.get_field('rho_background_i', iteration=it, slice_across='y')

    if xs is None:
        xs, zs = info.x, info.z

    B2      = np.maximum(Bx**2 + By**2 + Bz**2, 1e-12)
    n       = np.abs(rho + rho_bg) / sc.e
    Ti_J = 100 * sc.eV
    beta_th = n * Ti_J / (B2 / (2 * MU0))

    beta_frames.append(gaussian_filter(np.log10(np.maximum(beta_th, 1e-3)), sigma=1))
    #beta_frames.append(np.log10(np.maximum(beta_th, 1e-12)))

print("\nDone.")

# ── Animate ──────────────────────────────────────────────────────
N = 128
fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(beta_frames[0], origin='lower',
               extent=[xs[0], xs[-1], zs[0], zs[-1]],
               vmin=-3, vmax=3, cmap='RdBu_r', aspect='equal')
contour_handle = [ax.contour(xs, zs, beta_frames[0], levels=[0.0],
                             colors='k', linewidths=1.0)]
plt.axvline(0.6819, ymin=-2.5, ymax=2.5, label='r_CF')
plt.colorbar(im, ax=ax, label='log₁₀ β_th')
ax.set_xlabel('x (m)')
ax.set_ylabel('z (m)')
title = ax.set_title('')

def update(k):
    it = iterations[k]
    im.set_data(beta_frames[k])
    contour_handle[0].remove()
    contour_handle[0] = ax.contour(xs, zs, beta_frames[k], levels=[0.0],
                                   colors='k', linewidths=1.0)
    t_us = series_f.t[k] * 1e6
    title.set_text(f'β_th  step {it}  t = {t_us:.2f} µs')
    return [im]

ani = animation.FuncAnimation(fig, update, frames=len(iterations)-1, interval=100)
ani.save(f'{save_path}/beta_th_timelapse.mp4', writer='ffmpeg', fps=10, dpi=300)
plt.close()
print(f"Saved: {save_path}/beta_th_timelapse.mp4")

# B Lineout over time

In [ ]:
# ── Precompute ───────────────────────────────────────────────────
B_lineouts = []
xs_line    = None

for k, it in enumerate(iterations[1:]):
    print(f"  Loading {k+1}/{len(iterations)}", end='\r')

    Bx, info = series_f.get_field('B', coord='x', iteration=it, slice_across=['y', 'z'])
    By, _    = series_f.get_field('B', coord='y', iteration=it, slice_across=['y', 'z'])
    Bz, _    = series_f.get_field('B', coord='z', iteration=it, slice_across=['y', 'z'])

    if xs_line is None:
        xs_line = info.x

    B_mag = np.sqrt(Bx**2 + By**2 + Bz**2)
    B_lineouts.append(gaussian_filter(B_mag, sigma=1))

print("\nDone.")

# ── Animate ──────────────────────────────────────────────────────
B_max = max(b.max() for b in B_lineouts)

fig, ax = plt.subplots(figsize=(7, 4))
line,  = ax.plot(xs_line, B_lineouts[0], color='steelblue', lw=2)
ax.axhline(0, color='k', lw=0.5, ls='--')
ax.set_ylim(0, B_max * 1.1)
ax.set_xlabel('x (m)')
ax.set_ylabel('|B| (T)')
title = ax.set_title('')

def update(k):
    line.set_ydata(B_lineouts[k])
    t_us = series_f.t[k] * 1e6
    title.set_text(f'|B| lineout  step {iterations[k]}  t = {t_us:.2f} µs')
    return [line]

ani = animation.FuncAnimation(fig, update, frames=len(iterations[1:]), interval=100)
ani.save(f'{save_path}/B_lineout_timelapse.mp4', writer='ffmpeg', fps=10, dpi=150)
plt.close()
print(f"Saved: {save_path}/B_lineout_timelapse.mp4")



# Streamplots

In [ ]:
# Precompute Streamplots
B_streamplots = []
xs_line = None
xz_line = None

for k, it in enumerate(iterations[1:]):
    print(f"  Loading {k+1}/{len(iterations[1:])}", end='\r')
    Bx, info = series_f.get_field('B', coord='x', iteration=it, slice_across=['y'])
    By, _    = series_f.get_field('B', coord='y', iteration=it, slice_across=['y'])
    Bz, _    = series_f.get_field('B', coord='z', iteration=it, slice_across=['y'])

    xs_line = info.x
    xz_line = info.z 

    B_mag = np.sqrt(Bx**2 + By**2 + Bz**2)
    B_streamplots.append([Bx, Bz, B_mag])

print("\nDone")

# Animate
B_max = max(b[2].max() for b in B_streamplots)


N = 128
fig, ax = plt.subplots(figsize=(6, 6))
im = ax.streamplot(
    x=info.x,
    y=info.z,
    u=B_streamplots[0][0],
    v=B_streamplots[0][1],
    color=B_streamplots[0][2],
    cmap="viridis", density=2.0, linewidth=1,
    broken_streamlines=False,
)

ax.set_xlabel('x (m)')
ax.set_ylabel('z (m)')
title = ax.set_title('|B| Iteration 0')

def update(k):
    it = iterations[k]
    ax.cla()
    im = ax.streamplot(
        x=info.x,
        y=info.z,
        u=B_streamplots[k][0],
        v=B_streamplots[k][1],
        color=B_streamplots[k][2],
        cmap="viridis", density=1.2, linewidth=1
    )
    t_us = series_f.t[k] * 1e6
    ax.set_title(f'|B|  step {it}  t = {t_us:.2f} µs')
    ax.set_xlabel('x (m)')
    ax.set_ylabel('z (m)')
    return [im]

ani = animation.FuncAnimation(fig, update, frames=len(iterations)-1, interval=100)
ani.save(f'{save_path}/b_stream_timelapse.mp4', writer='ffmpeg', fps=10, dpi=300)
plt.close()
print(f"Saved: {save_path}/b_stream_timelapse.mp4")

# Face Cusp Losses over time (make a copy if running mid run)

In [ ]:
data   = np.load(f'{save_path}/cusp_flux.npz', allow_pickle=True)
times  = data['times'] * 1e6          # convert to µs
flux_minus = data['flux_minus']               # shape (n_steps, 6)
flux_plus = data['flux_plus']
#labels = data['face_labels']

total_loss = flux_minus

# ── Total loss rate vs time ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(times, total_loss, color='steelblue', lw=1.5, label='flux minus')
ax.set_xlabel('time (µs)')
ax.set_ylabel('particles lost per step')
ax.set_title('Cusp loss rate vs time')
ax.legend()
plt.tight_layout()
plt.savefig(f'{save_path}/cusp_loss_total.png', dpi=150)
plt.show()

# # ── Per-face loss vs time ─────────────────────────────────────────
# fig, ax = plt.subplots(figsize=(8, 4))
# for i, label in enumerate(labels):
#     ax.plot(times, losses[:, i], lw=1, label=label)
# ax.set_xlabel('time (µs)')
# ax.set_ylabel('particles lost per step')
# ax.set_title('Per-face cusp loss vs time')
# ax.legend()
# plt.tight_layout()
# plt.savefig('diags/cusp_loss_per_face.png', dpi=150)
# plt.show()

# Streamplot on top of beta

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import scipy.constants as sc
from scipy.ndimage import gaussian_filter
from openpmd_viewer import OpenPMDTimeSeries

MU0 = sc.mu_0

# ── Configuration ────────────────────────────────────────────────
save_path  = "diags"
#series_f   = OpenPMDTimeSeries("diags/field_diag/")
iterations = series_f.iterations

# ── Merged Precompute Loop ───────────────────────────────────────
beta_frames  = []
B_streamplots = []
xs = zs = None

for k, it in enumerate(iterations[1:]):
    print(f"  Loading {k+1}/{len(iterations)-1}", end='\r')

    Bx, info = series_f.get_field('B', coord='x', iteration=it, slice_across='y')
    By, _    = series_f.get_field('B', coord='y', iteration=it, slice_across='y')
    Bz, _    = series_f.get_field('B', coord='z', iteration=it, slice_across='y')
    rho,    _ = series_f.get_field('rho_stream_i',     iteration=it, slice_across='y')
    rho_bg, _ = series_f.get_field('rho_background_i', iteration=it, slice_across='y')

    if xs is None:
        xs, zs = info.x, info.z

    # Beta frame
    B2     = np.maximum(Bx**2 + By**2 + Bz**2, 1e-12)
    n      = np.abs(rho + rho_bg) / sc.e
    Ti_J   = 100 * sc.eV
    beta_th = n * Ti_J / (B2 / (2 * MU0))
    beta_frames.append(gaussian_filter(np.log10(np.maximum(beta_th, 1e-3)), sigma=1))

    # Streamplot frame (Bx, Bz in the xz plane)
    B_mag = np.sqrt(Bx**2 + Bz**2)
    B_streamplots.append((Bx, Bz, B_mag))

print("\nDone.")

# ── Helper: remove streamplot artists ───────────────────────────
def _clear_streamplot(sp):
    sp.lines.remove()


# ── Figure Setup ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 6))
ax2 = ax.inset_axes([0, 0, 1, 1])
ax2.set_axis_off()
ax2.patch.set_alpha(0)

# Layer 0: beta heatmap
im = ax.imshow(
    beta_frames[0], origin='lower',
    extent=[xs[0], xs[-1], zs[0], zs[-1]],
    vmin=-3, vmax=3, cmap='RdBu_r', aspect='equal', zorder=0
)

# Layer 1: beta=1 contour
contour_handle = [ax.contour(
    xs, zs, beta_frames[0], levels=[0.0],
    colors='k', linewidths=1.0, zorder=1
)]

seed_x = np.linspace(0.9,  1.2, 40)
seed_z = np.linspace(zs[0], zs[-1], 40)
seeds = np.array(np.meshgrid(seed_x, seed_z)).reshape(2, -1).T
# Layer 2: B-field streamplot on top
stream_handle = [ax2.streamplot(
    xs, zs,
    B_streamplots[0][0], B_streamplots[0][1],
    color=B_streamplots[0][2], cmap='hot', density=1.2, linewidth=0.8, zorder=2,
    start_points=seeds
)]

plt.colorbar(im, ax=ax, label='log₁₀ β_th')
ax.set_xlabel('x (m)')
ax.set_ylabel('z (m)')
ax.legend(loc='upper right', fontsize=8)
title = ax.set_title('')

# ── Animation Update ─────────────────────────────────────────────
def update(k):
    it = iterations[k + 1]
    ax2.cla()
    ax2.set_axis_off()
    ax2.patch.set_alpha(0)
    ax2.set_xlim(ax.get_xlim())
    ax2.set_ylim(ax.get_ylim())

    # Update beta heatmap
    im.set_data(beta_frames[k])

    # Update beta=1 contour
    contour_handle[0].remove()
    contour_handle[0] = ax.contour(
        xs, zs, beta_frames[k], levels=[0.0],
        colors='k', linewidths=1.0, zorder=1
    )

    stream_handle[0] = ax2.streamplot(
        xs, zs,
        B_streamplots[k][0], B_streamplots[k][1],
        color=B_streamplots[k][2], cmap='hot',  density=1.2, linewidth=0.8, zorder=2,
        start_points=seeds
    )

    t_us = series_f.t[k + 1] * 1e6
    title.set_text(f'β_th  step {it}  t = {t_us:.2f} µs')
    return [im]

# ── Save ─────────────────────────────────────────────────────────
ani = animation.FuncAnimation(
    fig, update, frames=len(iterations) - 1, interval=100
)
ani.save(f'{save_path}/beta_stream_overlay_localized.mp4', writer='ffmpeg', fps=10, dpi=300)
plt.close()
print(f"Saved: {save_path}/beta_stream_overlay.mp4")

# Bx lineout 

In [ ]:
# ── Precompute ───────────────────────────────────────────────────
B_lineouts = []
xs_line    = None

for k, it in enumerate(iterations[1:]):
    print(f"  Loading {k+1}/{len(iterations)}", end='\r')

    Bx, info = series_f.get_field('B', coord='x', iteration=it, slice_across=['y', 'z'])
    By, _    = series_f.get_field('B', coord='y', iteration=it, slice_across=['y', 'z'])
    Bz, _    = series_f.get_field('B', coord='z', iteration=it, slice_across=['y', 'z'])

    if xs_line is None:
        xs_line = info.x

    # B_mag = np.sqrt(Bx**2 + By**2 + Bz**2)
    # B_lineouts.append(gaussian_filter(B_mag, sigma=1))
    B_lineouts.append(Bx)

print("\nDone.")

# ── Animate ──────────────────────────────────────────────────────
B_max = max(b.max() for b in B_lineouts)
B_min = min(b.min() for b in B_lineouts)

fig, ax = plt.subplots(figsize=(7, 4))
line,  = ax.plot(xs_line, B_lineouts[0], color='steelblue', lw=2)
ax.axhline(0, color='k', lw=0.5, ls='--')
ax.set_ylim(B_min * 1.1, B_max * 1.1)
ax.set_xlabel('x (m)')
ax.set_ylabel('Bx (T)')
title = ax.set_title('')

def update(k):
    line.set_ydata(B_lineouts[k])
    t_us = series_f.t[k] * 1e6
    title.set_text(f'Bx lineout  step {iterations[k]}  t = {t_us:.2f} µs')
    return [line]

ani = animation.FuncAnimation(fig, update, frames=len(iterations[1:]), interval=100)
ani.save(f'{save_path}/Bx_lineout_timelapse.mp4', writer='ffmpeg', fps=10, dpi=150)
plt.close()
print(f"Saved: {save_path}/Bx_lineout_timelapse.mp4")

# Cusp losses time-synced

In [ ]:
data   = np.load(f'{save_path}/cusp_flux.npz', allow_pickle=True)
times  = data['times'] * 1e6          # convert to µs
flux_minus = data['flux_minus']               # shape (n_steps, 6)
flux_plus = data['flux_plus']
#labels = data['face_labels']

total_loss = flux_minus
t_k = [None]
# ── Total loss rate vs time ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(times, total_loss, color='steelblue', lw=1.5, label='flux minus')
diag_times = series_f.t * 1e6
t_k[0] = ax.axvline(diag_times[0], color='red')
ax.set_xlabel('time (µs)')
ax.set_ylabel('particles lost per step')
ax.set_title('Cusp loss rate vs time')
ax.legend()
plt.tight_layout()
plt.savefig(f'{save_path}/cusp_loss_total.png', dpi=150)
plt.show()

def update(k):
    t_k[0].remove()
    t_k[0] = ax.axvline(diag_times[k], color='red')

ani = animation.FuncAnimation(fig, update, frames=len(iterations), interval=100)
ani.save(f'{save_path}/cusp_loss_time_synced.mp4', writer='ffmpeg', fps=10, dpi=150)
plt.close()
print(f"Saved: {save_path}/cusp_loss_time_synced.mp4")

# Density over B-lines

In [ ]:
MU0 = sc.mu_0

# ── Configuration ────────────────────────────────────────────────
iterations = series_f.iterations

# ── Merged Precompute Loop ───────────────────────────────────────
density_frames  = []
B_streamplots = []
xs = zs = None

for k, it in enumerate(iterations[1:]):
    print(f"  Loading {k+1}/{len(iterations)-1}", end='\r')

    Bx, info = series_f.get_field('B', coord='x', iteration=it, slice_across='y')
    By, _    = series_f.get_field('B', coord='y', iteration=it, slice_across='y')
    Bz, _    = series_f.get_field('B', coord='z', iteration=it, slice_across='y')
    rho,    _ = series_f.get_field('rho_stream_i',     iteration=it, slice_across='y')
    rho_bg, _ = series_f.get_field('rho_background_i', iteration=it, slice_across='y')

    if xs is None:
        xs, zs = info.x, info.z

    # Density
    density_frames.append(np.log((rho + rho_bg) / sc.elementary_charge))

    # Streamplot frame (Bx, Bz in the xz plane)
    B_mag = np.sqrt(Bx**2 + Bz**2)
    B_streamplots.append((Bx, Bz, B_mag))

print("\nDone.")

# ── Helper: remove streamplot artists ───────────────────────────
def _clear_streamplot(sp):
    sp.lines.remove()


# ── Figure Setup ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 6))
ax2 = ax.inset_axes([0, 0, 1, 1])
ax2.set_axis_off()
ax2.patch.set_alpha(0)

vmin = np.min([_rho.min() for _rho in density_frames])
vmax = np.max([_rho.max() for _rho in density_frames])

# Layer 0: beta heatmap
im = ax.imshow(
    density_frames[0], origin='lower',
    extent=[xs[0], xs[-1], zs[0], zs[-1]],
    vmin=vmin, vmax=vmax, cmap='plasma', aspect='equal', zorder=0
)

# # Layer 1: beta=1 contour
# contour_handle = [ax.contour(
#     xs, zs, density_frames[0], levels=[0.0],
#     colors='k', linewidths=1.0, zorder=1
# )]
seeds = np.array(np.meshgrid(seed_x, seed_z)).reshape(2, -1).T
# Layer 2: B-field streamplot on top
stream_handle = [ax2.streamplot(
    xs, zs,
    B_streamplots[0][0], B_streamplots[0][1],
    color=B_streamplots[0][2], cmap='hot', density=1.2, linewidth=0.8, zorder=2,
)]

plt.colorbar(im, ax=ax, label='log₁₀ rho/q')
ax.set_xlabel('x (m)')
ax.set_ylabel('z (m)')
ax.legend(loc='upper right', fontsize=8)
title = ax.set_title('')

# ── Animation Update ─────────────────────────────────────────────
def update(k):
    it = iterations[k + 1]
    ax2.cla()
    ax2.set_axis_off()
    ax2.patch.set_alpha(0)
    ax2.set_xlim(ax.get_xlim())
    ax2.set_ylim(ax.get_ylim())

    # Update density map
    im.set_data(density_frames[k])

    # Update beta=1 contour
    # contour_handle[0].remove()
    # contour_handle[0] = ax.contour(
    #     xs, zs, beta_frames[k], levels=[0.0],
    #     colors='k', linewidths=1.0, zorder=1
    # )

    stream_handle[0] = ax2.streamplot(
        xs, zs,
        B_streamplots[k][0], B_streamplots[k][1],
        color=B_streamplots[k][2], cmap='hot',  density=1.2, linewidth=0.8, zorder=2,
    )

    t_us = series_f.t[k + 1] * 1e6
    title.set_text(f'rho/q  step {it}  t = {t_us:.2f} µs')
    return [im]

# ── Save ─────────────────────────────────────────────────────────
ani = animation.FuncAnimation(
    fig, update, frames=len(iterations) - 1, interval=100
)
ani.save(f'{save_path}/density_stream_overlay_localized.mp4', writer='ffmpeg', fps=10, dpi=300)
plt.close()
print(f"Saved: {save_path}/density_stream_overlay.mp4")